In [0]:
%sql
drop table if exists Car;
create table Car
(
  brand string,
  name string,
  price decimal
)


In [0]:
%sql
insert into Car values
  ('Toyota', 'Corolla', 20000.00),
  ('Toyota', 'Camry', 25000.00),
  ('Honda', 'Civic', 21000.00),
  ('Honda', 'Accord', 26000.00),
  ('Ford', 'Focus', 19000.00),
  ('Ford', 'Fusion', 23000.00),
  ('Toyota', 'Yaris', 17000.00),
  ('Honda', 'Fit', 18000.00);

num_affected_rows,num_inserted_rows
8,8


In [0]:
%sql
select * from Car;

brand,name,price
Toyota,Corolla,20000
Toyota,Camry,25000
Honda,Civic,21000
Honda,Accord,26000
Ford,Focus,19000
Ford,Fusion,23000
Toyota,Yaris,17000
Honda,Fit,18000


In [0]:
%sql
select *,
row_number() over(order by price asc)
from Car;

brand,name,price,row_number()OVER(ORDERBYpriceASC)
Toyota,Yaris,17000,1
Honda,Fit,18000,2
Ford,Focus,19000,3
Toyota,Corolla,20000,4
Honda,Civic,21000,5
Ford,Fusion,23000,6
Toyota,Camry,25000,7
Honda,Accord,26000,8


In [0]:
%sql
select *,
row_number() over(partition by brand order by price asc)
from Car;

brand,name,price,row_number()OVER(PARTITIONBYbrandORDERBYpriceASC)
Ford,Focus,19000,1
Ford,Fusion,23000,2
Honda,Fit,18000,1
Honda,Civic,21000,2
Honda,Accord,26000,3
Toyota,Yaris,17000,1
Toyota,Corolla,20000,2
Toyota,Camry,25000,3


In [0]:
%sql
select *,
lead(name) over(order by name )
from Car;

brand,name,price,lead(name)OVER(ORDERBYname)
Honda,Accord,26000,Camry
Toyota,Camry,25000,Civic
Honda,Civic,21000,Corolla
Toyota,Corolla,20000,Fit
Honda,Fit,18000,Focus
Ford,Focus,19000,Fusion
Ford,Fusion,23000,Yaris
Toyota,Yaris,17000,null


In [0]:
%sql
select *,
lead(name) over(partition by brand order by name asc)
from Car;

brand,name,price,lead(name)OVER(PARTITIONBYbrandORDERBYnameASC)
Ford,Focus,19000,Fusion
Ford,Fusion,23000,null
Honda,Accord,26000,Civic
Honda,Civic,21000,Fit
Honda,Fit,18000,null
Toyota,Camry,25000,Corolla
Toyota,Corolla,20000,Yaris
Toyota,Yaris,17000,null


In [0]:
%sql
select *,
sum(price)over()
from Car;

brand,name,price,sum(price)OVER()
Toyota,Corolla,20000,169000
Toyota,Camry,25000,169000
Honda,Civic,21000,169000
Honda,Accord,26000,169000
Ford,Focus,19000,169000
Ford,Fusion,23000,169000
Toyota,Yaris,17000,169000
Honda,Fit,18000,169000


In [0]:
%sql
select *,
sum(price) over(rows between unbounded preceding and current row)
from Car;

brand,name,price,sum(price)OVER(ROWSBETWEENUNBOUNDEDPRECEDINGANDCURRENTROW)
Toyota,Corolla,20000,20000
Toyota,Camry,25000,45000
Honda,Civic,21000,66000
Honda,Accord,26000,92000
Ford,Focus,19000,111000
Ford,Fusion,23000,134000
Toyota,Yaris,17000,151000
Honda,Fit,18000,169000


In [0]:
%sql
select *,
sum(price) over(rows between current row and unbounded following)
from Car;

brand,name,price,sum(price)OVER(ROWSBETWEENCURRENTROWANDUNBOUNDEDFOLLOWING)
Toyota,Corolla,20000,169000
Toyota,Camry,25000,149000
Honda,Civic,21000,124000
Honda,Accord,26000,103000
Ford,Focus,19000,77000
Ford,Fusion,23000,58000
Toyota,Yaris,17000,35000
Honda,Fit,18000,18000


In [0]:
df=spark.sql('select * from Car')
df.display()

brand,name,price
Toyota,Corolla,20000
Toyota,Camry,25000
Honda,Civic,21000
Honda,Accord,26000
Ford,Focus,19000
Ford,Fusion,23000
Toyota,Yaris,17000
Honda,Fit,18000


In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import *

df.withColumn('R'
    ,row_number().over(Window.orderBy(col('price')))).display()

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1061: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


brand,name,price,R
Toyota,Yaris,17000,1
Honda,Fit,18000,2
Ford,Focus,19000,3
Toyota,Corolla,20000,4
Honda,Civic,21000,5
Ford,Fusion,23000,6
Toyota,Camry,25000,7
Honda,Accord,26000,8


In [0]:
df.withColumn(
    'R',
    row_number().over(Window.partitionBy('brand').orderBy('price'))
).display()

brand,name,price,R
Ford,Focus,19000,1
Ford,Fusion,23000,2
Honda,Fit,18000,1
Honda,Civic,21000,2
Honda,Accord,26000,3
Toyota,Yaris,17000,1
Toyota,Corolla,20000,2
Toyota,Camry,25000,3


In [0]:
df.withColumn(
    'L',
    lead(col('name')).over(Window.partitionBy(col('brand')).orderBy(col('name')))
).display()

brand,name,price,L
Ford,Focus,19000,Fusion
Ford,Fusion,23000,null
Honda,Accord,26000,Civic
Honda,Civic,21000,Fit
Honda,Fit,18000,null
Toyota,Camry,25000,Corolla
Toyota,Corolla,20000,Yaris
Toyota,Yaris,17000,null


In [0]:
df.withColumn(
    'P',
    sum(col('price')).over(Window.partitionBy())
).display()

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1061: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


brand,name,price,P
Toyota,Corolla,20000,169000
Toyota,Camry,25000,169000
Honda,Civic,21000,169000
Honda,Accord,26000,169000
Ford,Focus,19000,169000
Ford,Fusion,23000,169000
Toyota,Yaris,17000,169000
Honda,Fit,18000,169000


In [0]:
df.withColumn(
    'P',
    sum(col('price')).over(Window.partitionBy().rowsBetween(Window.unboundedPreceding,Window.currentRow))
).display()

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1061: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


brand,name,price,P
Toyota,Corolla,20000,20000
Toyota,Camry,25000,45000
Honda,Civic,21000,66000
Honda,Accord,26000,92000
Ford,Focus,19000,111000
Ford,Fusion,23000,134000
Toyota,Yaris,17000,151000
Honda,Fit,18000,169000


In [0]:
df.withColumn(
    'P',
    sum(col('price')).over(Window.partitionBy().rowsBetween(Window.currentRow,Window.unboundedFollowing))
).display()

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1061: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


brand,name,price,P
Toyota,Corolla,20000,169000
Toyota,Camry,25000,149000
Honda,Civic,21000,124000
Honda,Accord,26000,103000
Ford,Focus,19000,77000
Ford,Fusion,23000,58000
Toyota,Yaris,17000,35000
Honda,Fit,18000,18000


In [0]:
df.withColumn(
    'P',
    sum(col('price')).over(Window.partitionBy().rowsBetween(Window.unboundedPreceding,Window.unboundedFollowing))
).display()

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1061: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


brand,name,price,P
Toyota,Corolla,20000,169000
Toyota,Camry,25000,169000
Honda,Civic,21000,169000
Honda,Accord,26000,169000
Ford,Focus,19000,169000
Ford,Fusion,23000,169000
Toyota,Yaris,17000,169000
Honda,Fit,18000,169000
